In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

In [18]:
# === Model Definition ===

class SimpleTransformer(nn.Module):
    """A minimal transformer language model."""

    def __init__(self, vocab_size, d_model=128, n_heads=4, n_layers=2, max_seq_len=64):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.position_embedding = nn.Embedding(max_seq_len, d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=d_model * 4,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.output_proj = nn.Linear(d_model, vocab_size)
        self.max_seq_len = max_seq_len

    def forward(self, x):
        seq_len = x.shape[1]
        positions = torch.arange(seq_len, device=x.device).unsqueeze(0)

        # Embeddings
        x = self.token_embedding(x) + self.position_embedding(positions)

        # Causal mask for autoregressive modeling
        mask = nn.Transformer.generate_square_subsequent_mask(seq_len, device=x.device)

        # Transformer layers
        x = self.transformer(x, mask=mask)

        # Project to vocabulary
        logits = self.output_proj(x)
        return logits


# === Dataset ===

class TextDataset(Dataset):
    """Simple character-level dataset."""

    def __init__(self, text, seq_len=64):
        self.seq_len = seq_len

        # Build vocabulary from text
        self.chars = sorted(list(set(text)))
        self.vocab_size = len(self.chars)
        self.char_to_idx = {c: i for i, c in enumerate(self.chars)}
        self.idx_to_char = {i: c for c, i in self.char_to_idx.items()}

        # Encode full text
        self.data = torch.tensor([self.char_to_idx[c] for c in text], dtype=torch.long)

    def __len__(self):
        return len(self.data) - self.seq_len

    def __getitem__(self, idx):
        chunk = self.data[idx:idx + self.seq_len + 1]
        x = chunk[:-1]
        y = chunk[1:]
        return x, y

    def decode(self, indices):
        return ''.join([self.idx_to_char[i.item()] for i in indices])


# === Training ===

def compute_loss(model, x, y):
    """Compute cross-entropy loss for next-token prediction."""
    logits = model(x)

    # Reshape for cross entropy: (batch * seq_len, vocab_size) vs (batch * seq_len,)
    batch_size, seq_len, vocab_size = logits.shape
    loss = F.cross_entropy(
        logits.view(batch_size * seq_len, vocab_size),
        y.view(batch_size * seq_len),
        reduction='mean'
    )
    return loss


def generate_sample(model, dataset, prompt="The ", max_tokens=50):
    """Generate text from the model."""
    model.eval()

    # Encode prompt
    tokens = [dataset.char_to_idx.get(c, 0) for c in prompt]
    x = torch.tensor(tokens, dtype=torch.long).unsqueeze(0)
    generated_tokens = []

    if torch.cuda.is_available():
        x = x.cuda()

    # Generate tokens
    with torch.no_grad():
        for _ in range(max_tokens):
            # Get prediction for last position
            logits = model(x)
            next_token_logits = logits[0, -1, :]

            # Sample from distribution
            probs = F.softmax(next_token_logits, dim=-1)
            next_token = torch.multinomial(probs, 1)

            # Store generated token
            generated_tokens.append(next_token.item())

            # Append and continue (sliding window for model input)
            x = torch.cat([x, next_token.unsqueeze(0)], dim=1)
            if x.shape[1] > dataset.seq_len:
                x = x[:, -dataset.seq_len:]

    # Return prompt + generated text
    generated_text = ''.join([dataset.idx_to_char[t] for t in generated_tokens])
    return prompt + generated_text


def train(model, train_loader, val_loader, optimizer, scheduler, num_epochs=5):
    """Main training loop."""

    for epoch in range(num_epochs):
        model.train()

        total_train_loss = 0
        total_val_loss = 0
        num_batches = 0

        for batch_idx, (x, y) in enumerate(train_loader):
            if torch.cuda.is_available():
                x, y = x.cuda(), y.cuda()

            # Forward pass
            loss = compute_loss(model, x, y)

            # Backward pass
            optimizer.zero_grad()
            loss.backward()

            # Update weights
            optimizer.step()
            
            total_train_loss += loss.item()
            num_batches += 1

        scheduler.step()
        avg_train_loss = total_train_loss / num_batches

        # --- Validation Phase ---
        model.eval()
        total_val_loss = 0.0

        with torch.no_grad():
            for batch_idx, (x, y) in enumerate(val_loader):
                if torch.cuda.is_available():
                    x, y = x.cuda(), y.cuda()
                loss = compute_loss(model, x, y)
                total_val_loss += loss.item()
        avg_val_loss = total_val_loss / len(val_loader)

        print(f"Epoch {epoch + 1}/{num_epochs}, Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}, LR: {scheduler.get_last_lr()[0]:.6f}")
   
    return model

In [19]:
def main():
    # Sample training text (simple patterns that should be easy to learn)
    training_text = """
    The quick brown fox jumps over the lazy dog. The lazy dog sleeps all day.
    A quick brown fox is faster than a lazy dog. The dog is lazy but happy.
    Quick foxes jump over lazy dogs. The brown fox is very quick indeed.
    """ * 100  # Repeat to get enough data

    # Create dataset and dataloader
    dataset = TextDataset(training_text, seq_len=32)
    #train_loader = DataLoader(dataset, batch_size=32, shuffle=True)

    print(f"Vocabulary size: {dataset.vocab_size}")
    print(f"Dataset size: {len(dataset)} sequences")
    print(f"Sample characters: {dataset.chars[:20]}")

    n_train = int(0.9 * len(dataset))
    n_val = len(dataset) - n_train
    train_dataset, val_dataset = torch.utils.data.random_split(dataset, [n_train, n_val])
    train_loader = DataLoader(train_dataset, batch_size = 32, shuffle = True)
    val_loader = DataLoader(val_dataset, batch_size = 32, shuffle = True)


    # Initialize model
    model = SimpleTransformer(
        vocab_size=dataset.vocab_size,
        d_model=64,
        n_heads=4,
        n_layers=2,
        max_seq_len=32
    )

    if torch.cuda.is_available():
        model = model.cuda()
        print("Using CUDA")

    # Count parameters
    num_params = sum(p.numel() for p in model.parameters())
    print(f"Model parameters: {num_params:,}")

    # Optimizer and scheduler
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.01)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=5  # Expects 5 scheduler.step() calls (one per epoch)
    )

    # Train
    print("\nStarting training...")
    print("-" * 50)
    model = train(model, train_loader, val_loader, optimizer, scheduler, num_epochs=5)

    # Generate samples
    print("-" * 50)
    print("\nGenerated samples:")
    for prompt in ["The quick ", "A lazy ", "The dog "]:
        sample = generate_sample(model, dataset, prompt=prompt, max_tokens=50)
        print(f"  '{prompt}' -> '{sample}'")


if __name__ == "__main__":
    main()

Vocabulary size: 32
Dataset size: 23168 sequences
Sample characters: ['\n', ' ', '.', 'A', 'Q', 'T', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n']
Model parameters: 106,144

Starting training...
--------------------------------------------------
Epoch 1/5, Train Loss: 0.4652, Val Loss: 0.0900, LR: 0.000905
Epoch 2/5, Train Loss: 0.1066, Val Loss: 0.0836, LR: 0.000655
Epoch 3/5, Train Loss: 0.0938, Val Loss: 0.0805, LR: 0.000345
Epoch 4/5, Train Loss: 0.0888, Val Loss: 0.0796, LR: 0.000095
Epoch 5/5, Train Loss: 0.0860, Val Loss: 0.0778, LR: 0.000000
--------------------------------------------------

Generated samples:
  'The quick ' -> 'The quick brown fox jumps over the lazy dog. The lazy dog sl'
  'A lazy ' -> 'A lazy dogs. The brown fox is very quick indeed.
    
   '
  'The dog ' -> 'The dog is lazy but happy.
    Quick foxes jump over lazy '
